In [51]:
import requests
import pandas as pd
from datetime import datetime
from google.transit import gtfs_realtime_pb2


URL = "https://realtime.gtfs.de/realtime-free.pb"

response = requests.get(URL, timeout=30)


In [52]:
def parse_gtfs_feed(response):
    """
    Parse a GTFS-RT response into a FeedMessage.

    Parameters
    ----------
    response : requests.Response
        HTTP response containing the serialized GTFS-RT feed.

    Returns
    -------
    FeedMessage
        Parsed GTFS-RT feed.
    """
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(response.content)

    print(f"Number of entities: {len(feed.entity)}")

    return feed

feed = parse_gtfs_feed(response)


Number of entities: 119102


In [53]:
for entity in feed.entity[:10]:
    print(entity)

id: "658155tu"
trip_update {
  trip {
    trip_id: "658155"
    start_date: "20260905"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 0
    departure {
      delay: 0
      time: 1788611940
    }
    stop_id: "230052"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 1
    arrival {
      delay: 0
      time: 1788612060
    }
    departure {
      delay: 0
      time: 1788612060
    }
    stop_id: "528466"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 2
    arrival {
      delay: 0
      time: 1788612120
    }
    departure {
      delay: 0
      time: 1788612120
    }
    stop_id: "428151"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 3
    arrival {
      delay: 0
      time: 1788612180
    }
    departure {
      delay: 0
      time: 1788612180
    }
    stop_id: "108066"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence

In [56]:
def parse_trip_updates(feed, stop_names, trip_lines):
    """
    Parse GTFS-RT trip updates into a pandas DataFrame.

    Parameters
    ----------
    feed : FeedMessage
        Parsed GTFS-RT feed containing trip updates.
    stop_names : dict
        Mapping from stop IDs to stop names.
    trip_lines : dict
        Mapping from trip IDs to line names.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing trip, line, stop, arrival,
        departure, and delay information.
    """
    rows = []

    for entity in feed.entity:
        if not entity.HasField("trip_update"):
            continue

        trip = entity.trip_update.trip

        # Get line for this trip
        line = trip_lines.get(str(trip.trip_id))

        # Ignore trips that are not part of the selected agencies
        if line is None:
            continue

        for stop in entity.trip_update.stop_time_update:

            row = {
                "trip_id": trip.trip_id,
                "start_date": trip.start_date,
                "line": line,
                "stop_id": str(stop.stop_id),
                "stop_name": stop_names.get(str(stop.stop_id)),
                "stop_sequence": stop.stop_sequence,
            }

            if stop.HasField("departure"):
                row["departure_time"] = datetime.fromtimestamp(
                    stop.departure.time
                )
                row["departure_delay"] = stop.departure.delay

            if stop.HasField("arrival"):
                row["arrival_time"] = datetime.fromtimestamp(
                    stop.arrival.time
                )
                row["arrival_delay"] = stop.arrival.delay

            rows.append(row)

    return pd.DataFrame(rows)

In [57]:
def preprocess_gtfs(data_dir, munich_agencies):
    """
    Preprocess GTFS static data for selected agencies.

    Returns
    -------
    trip_lines : dict
        Mapping from trip_id to line name.

    stop_names : dict
        Mapping from stop_id to stop name.
    """

    routes_df = pd.read_csv(f"{data_dir}/routes.txt")
    trips_df = pd.read_csv(f"{data_dir}/trips.txt")
    stops_df = pd.read_csv(f"{data_dir}/stops.txt")

    routes_df["route_id"] = routes_df["route_id"].astype(str)
    routes_df["agency_id"] = routes_df["agency_id"].astype(str)

    trips_df["trip_id"] = trips_df["trip_id"].astype(str)
    trips_df["route_id"] = trips_df["route_id"].astype(str)

    stops_df["stop_id"] = stops_df["stop_id"].astype(str)

    munich_routes = routes_df[
        routes_df["agency_id"].isin(munich_agencies)
    ]

    route_lines = (
        munich_routes
        .set_index("route_id")["route_short_name"]
        .to_dict()
    )

    munich_trips = trips_df[
        trips_df["route_id"].isin(route_lines)
    ]

    trip_lines = (
        munich_trips
        .set_index("trip_id")["route_id"]
        .map(route_lines)
        .to_dict()
    )

    stop_names = (
        stops_df
        .set_index("stop_id")["stop_name"]
        .to_dict()
    )

    return trip_lines, stop_names


In [58]:
munich_agencies = ["100", "191", "364"]

trip_lines, stop_names = preprocess_gtfs(
    "../data",
    munich_agencies
)

In [59]:
df = parse_trip_updates(
    feed,
    stop_names,
    trip_lines
)

df.head(100)

,trip_id,start_date,line,stop_id,stop_name,stop_sequence,departure_time,departure_delay,arrival_time,arrival_delay
0,841957,20260905,U6,525855,"Colditz, Gewerbegebiet/anona",0,2026-09-05 17:06:00,0.0,NaT,NaN
1,1628519,20260905,63,583277,None,0,2026-09-05 13:35:00,0.0,NaT,NaN
2,1743480,20260905,271,568676,Leverkusen Mathildenhof Schöneberger Str.,0,2026-09-05 15:31:00,0.0,NaT,NaN
3,1845122,20260905,184,357527,"Ursberg, Gymnasium",0,2026-09-05 15:14:00,0.0,NaT,NaN
4,1845122,20260905,184,431872,Überl. Naturata,1,2026-09-05 15:15:00,0.0,2026-09-05 15:15:00,0.0
...,...,...,...,...,...,...,...,...,...,...
95,419240,20260905,190,478162,Burghausen Kapuzinergasse,29,2026-09-05 14:19:30,210.0,2026-09-05 14:19:30,210.0
96,419240,20260905,190,477055,Mettingen Cannstatter Str.,30,2026-09-05 14:20:30,210.0,2026-09-05 14:20:30,210.0
97,419240,20260905,190,79218,Bad Ems Pfingstwiese,31,2026-09-05 14:23:30,210.0,2026-09-05 14:23:30,210.0
98,419240,20260905,190,637126,"Bernburg Zickzackhausen, Richtung Sternzfeld",32,2026-09-05 14:26:30,210.0,2026-09-05 14:26:30,210.0


In [60]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33090 entries, 0 to 33089
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   trip_id          33090 non-null  object        
 1   start_date       33090 non-null  object        
 2   line             33090 non-null  object        
 3   stop_id          33090 non-null  object        
 4   stop_name        32506 non-null  object        
 5   stop_sequence    33090 non-null  int64         
 6   departure_time   29699 non-null  datetime64[ns]
 7   departure_delay  29699 non-null  float64       
 8   arrival_time     28882 non-null  datetime64[ns]
 9   arrival_delay    28882 non-null  float64       
dtypes: datetime64[ns](2), float64(2), int64(1), object(5)
memory usage: 2.5+ MB


In [61]:
def load_existing_realtime_data(
    parquet_path="../data/mvv_realtime.parquet"
):
    """
    Load existing MVV real-time data from a Parquet file.

    Parameters
    ----------
    parquet_path : str
        Path to the existing Parquet file.

    Returns
    -------
    pandas.DataFrame
        Existing MVV real-time data.
    """

    return pd.read_parquet(parquet_path)

In [62]:
def update_realtime_data(
    existing_df,
    new_df
):
    """
    Add new real-time data and keep the latest
    observation for each trip and stop.

    Parameters
    ----------
    existing_df : pandas.DataFrame
        Previously stored MVV real-time data.

    new_df : pandas.DataFrame
        Newly retrieved MVV real-time data.

    Returns
    -------
    pandas.DataFrame
        Updated MVV real-time data.
    """

    combined_df = pd.concat(
        [
            existing_df,
            new_df
        ],
        ignore_index=True
    )

    combined_df = (
        combined_df
        .drop_duplicates(
            subset=[
                "trip_id",
                "start_date",
                "stop_id"
            ],
            keep="last"
        )
        .reset_index(drop=True)
    )

    return combined_df

In [63]:
def save_realtime_data(
    realtime_df,
    parquet_path="../data/mvv_realtime.parquet"
):
    """
    Save MVV real-time data to a Parquet file.

    Parameters
    ----------
    realtime_df : pandas.DataFrame
        MVV real-time data to save.

    parquet_path : str
        Path where the Parquet file is stored.

    Returns
    -------
    None
        The DataFrame is saved to the specified Parquet file.
    """

    realtime_df.to_parquet(
        parquet_path,
        index=False
    )

In [64]:
existing_df = load_existing_realtime_data()

updated_df = update_realtime_data(
    existing_df,
    df
)

save_realtime_data(
    updated_df
)

In [65]:
existing_df = load_existing_realtime_data()


In [66]:
existing_df

,trip_id,start_date,line,stop_id,stop_name,stop_sequence,departure_time,departure_delay,arrival_time,arrival_delay
0,1192330,20260904,190,614970,August-Everding-Straße,1,2026-09-05 00:37:33,3.0,2026-09-05 00:37:33,3.0
1,1192330,20260904,190,576765,Sankt Pius,2,2026-09-05 00:37:48,-12.0,2026-09-05 00:37:48,-12.0
2,1192330,20260904,190,686743,Grafinger Straße,3,2026-09-05 00:39:22,-8.0,2026-09-05 00:38:52,-38.0
3,1192330,20260904,190,634241,Altöttinger Straße,4,2026-09-05 00:40:34,4.0,2026-09-05 00:40:11,-19.0
4,1192330,20260904,190,561261,Schlüsselbergstraße,5,2026-09-05 00:41:42,-18.0,2026-09-05 00:41:11,-49.0
...,...,...,...,...,...,...,...,...,...,...
49459,279582,20260905,822,196266,Holzkirchen,26,2026-09-05 13:22:34,-26.0,2026-09-05 13:22:22,-38.0
49460,279582,20260905,822,262947,Geroksruhe,27,2026-09-05 13:24:28,-32.0,2026-09-05 13:24:28,-32.0
49461,279582,20260905,822,151491,Donau-Arena,28,2026-09-05 13:27:46,46.0,2026-09-05 13:26:02,-58.0
49462,279582,20260905,822,443152,Limberg Gasthaus,29,2026-09-05 13:28:58,58.0,2026-09-05 13:28:42,42.0


In [67]:
existing_df.duplicated(
    subset=[
        "trip_id",
        "start_date",
        "stop_sequence"
    ]
).sum()


np.int64(3)